In [4]:
from google.colab import drive
from shutil import copy2
from duckdb import connect as dcon
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats
import plotly.express as px
from pylab import rcParams
import pandas as pd
import numpy as np
from warnings import filterwarnings

darkmodel = True
rcParams['figure.figsize'] = (12,6)
pd.options.display.float_format = '{:,.2f}'.format
filterwarnings('ignore', category=FutureWarning)

%matplotlib inline

if darkmodel:
    # 1. Define a sophisticated E-commerce color palette
    # These colors are chosen for high contrast against the #212946 background
    colors = [
        "#08F7FE",  # Cyan Glow
        "#FE53BB",  # Neon Pink
        "#F5D300",  # Cyber Yellow
        "#00ff41",  # Matrix Green
        "#9467bd",  # Royal Purple
    ]

    # 2. Enhanced Dictionary with Complex Styling
    refined_dark_style = {
        # Background and Canvas
        "figure.facecolor": "#212946",
        "axes.facecolor": "#212946",
        "savefig.facecolor": "#212946",

        # Grid Sophistication
        "axes.grid": True,
        "axes.grid.which": "both",
        "grid.color": "#2A3459",
        "grid.linewidth": "1",
        "grid.alpha": 0.5,

        # Typography & Labels (Optimized for readability)
        "text.color": "#E2E2E2",
        "axes.labelcolor": "#E2E2E2",
        "axes.labelsize": 14,
        "axes.titlesize": 18,
        "axes.titleweight": "bold",
        "axes.titlepad": 20,
        "xtick.color": "#8E9CC3",
        "ytick.color": "#8E9CC3",
        "font.size": 12,

        # Spines (Clean aesthetic)
        "axes.spines.left": False,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "axes.spines.bottom": True,
        "axes.edgecolor": "#2A3459",

        # Line & Marker Settings
        "lines.linewidth": 2.5,
        "lines.markersize": 8,
        "axes.prop_cycle": plt.cycler(color=colors),
    }

    plt.rcParams.update(refined_dark_style)

else:
    # 1. Defining the "Paper & Ink" Palette
    # Deep Blue #003366 | Oxide Red #A52A2A
    ecom_vintage_colors = [
        "#003366",  # Oxford Blue (Primary)
        "#A52A2A",  # Oxide Red (Comparison)
        "#006400",  # Dark Green (Success Metrics)
        "#704214",  # Sepia (Neutral)
    ]

    vintage_style = {
        # Background - The specific parchment hex you requested
        "figure.facecolor": "#f7e4b7",
        "axes.facecolor": "#f7e4b7",
        "savefig.facecolor": "#f7e4b7",

        # Grid - Subtle contrast using a darker version of the background
        "axes.grid": True,
        "grid.color": "#e2d1a8",
        "grid.linestyle": "-",
        "grid.linewidth": 1.0,

        # Typography - Deep Charcoal/Blue instead of pure black for a softer feel
        "text.color": "#2C2C2C",
        "axes.labelcolor": "#2C2C2C",
        "xtick.color": "#5D5D5D",
        "ytick.color": "#5D5D5D",
        "axes.titlesize": 16,
        "axes.titleweight": "bold",
        "axes.titlepad": 15,
        "font.size": 11,

        # Spines - Classic 'L-frame' for publication
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.spines.left": True,
        "axes.spines.bottom": True,
        "axes.edgecolor": "#2C2C2C",
        "axes.linewidth": 1.5,

        # Data Point Styling
        "axes.prop_cycle": plt.cycler(color=ecom_vintage_colors),
        "lines.linewidth": 2.2,
        "lines.markersize": 8,
        "patch.edgecolor": "#f7e4b7", # Borders on bars/pie slices
    }

    plt.rcParams.update(vintage_style)

In [5]:
drive.mount('/content/drive')
pd.set_option('display.max_rows', 50)

def ListFiles(Dirs):
    errormsg = f"Error: Directory '{Dirs}' does not exist or is not a directory."
    assert os.path.isdir(Dirs), errormsg
    file_data = list()
    for item in os.listdir(Dirs):
        item_path = os.path.join(Dirs, item)
        if os.path.isfile(item_path):
            try:
                size_bytes = os.path.getsize(item_path)
                size_mb = size_bytes / (1024 * 1024)  # Convert bytes to MB
                file_data.append({'File Name': item, 'Size (MB)': size_mb})
            except Exception as err:
                print(f"Could not get size for {item_path}: {err}")

    Files = pd.DataFrame(file_data)
    return Files

Mounted at /content/drive


In [6]:
MyFiles = ListFiles('/content/drive/MyDrive/Colab Notebooks')
DatFilename = MyFiles[~MyFiles['File Name'].str.contains('.ipynb', na = False)]
display(DatFilename)

,File Name,Size (MB)
77,MasterData.parquet,145.58
80,InstaCart.db,263.76
83,xgboost_ltr_model.json,2.43
84,catboost_ltr_model.cbm,0.32
85,lightgbm_ltr_model.txt,6.21
86,lgbm_optuna_ranker_model.txt,0.58
87,test_df.parquet,3.84


In [7]:
test_df = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/test_df.parquet')
display(test_df.head())

,order_id,user_id,product_id,aisle_id,department_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,add_to_cart_order,reordered,user_total_orders,user_avg_days_between,user_avg_cart_pos,user_total_reorders,index,prod_total_reorders,prod_reorder_rate,prod_order_count,prod_avg_cart_pos
0,252513,110001,41588,14,20,38,4,19,3.00,8,1,38,3.00,6.50,9,32736,256,0.65,391,8.82
1,252513,110001,49628,120,16,38,4,19,3.00,12,1,38,3.00,6.50,9,39078,123,0.66,186,8.49
2,252654,42214,23178,98,7,10,5,16,12.00,3,0,10,12.00,2.00,1,18299,108,0.61,176,7.26
3,252654,42214,24838,91,16,10,5,16,12.00,1,0,10,12.00,2.00,1,19563,1615,0.74,2169,6.28
4,252654,42214,46667,83,4,10,5,16,12.00,2,1,10,12.00,2.00,1,36739,1241,0.62,2007,9.49


In [8]:
import pandas as pd

TheFeature = ['order_number', 'order_dow', 'days_since_prior_order',
       'add_to_cart_order', 'user_total_orders', 'user_avg_days_between',
       'user_avg_cart_pos', 'user_total_reorders', 'prod_total_reorders',
       'prod_reorder_rate', 'prod_order_count', 'prod_avg_cart_pos']

TheFeatures = pd.Index(TheFeature)

In [9]:
def Final_LTRpreps(Data : pd.DataFrame, Feature = list):
    df_sorted = Data.sort_values(['user_id']).reset_index(drop=True)
    X = df_sorted[Feature].values
    y = df_sorted['reordered'].values
    query_ids = df_sorted['user_id'].values
    # Hitung group (jumlah sampel per user)
    group = df_sorted.groupby('user_id').size().values
    return X, y, group, query_ids

In [10]:
X_test, y_test, group_test, q_test = Final_LTRpreps(test_df, TheFeatures)